In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
from matplotlib.pyplot import cm
import seaborn as sns
from matplotlib import rc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def interpolate_pole(time, lons, lats, array):
    
    #This function interpolates an array to 90°N and 90°S latitude and 360° longitude
    
    # extent latitude and longitude vector
    lats_90=np.zeros((len(lats)+2))
    
    for lat in range(1,len(lats)+1):
        lats_90[lat]=lats[lat-1]
        
    if lats[0]>0:    
        lats_90[0]=90    
        lats_90[len(lats)+1]=-90
        
    if lats[0]<0:      
        lats_90[0]=-90    
        lats_90[len(lats)+1]=90
    
    lons_360=np.zeros((len(lons)+1))
    
    for lon in range(len(lons)):
        lons_360[lon]=lons[lon]
        
    lons_360[len(lons)]=360

    #interpolate array to new grid points
    
    array_interpolated=np.zeros((len(lats)+2,len(lons)+1))
    array_interpolated[1:len(lats)+1,0:len(lons)]=array
    array_interpolated[1:len(lats)+1,len(lons)]=array[:,0]
    array_interpolated[0,:]=np.mean(array[0,:])
    array_interpolated[len(lats)+1,:]=np.mean(array[len(lats)-1,:])
        
    return lats_90, lons_360, array_interpolated  
  


def interpolate_lons(time, lons, lats, array):
    
    # This function interpolates an array to 360° longitude
    
    lats_90 = lats
    
    # extent longitude vector 
    
    lons_360=np.zeros((len(lons)+1))
    
    for lon in range(len(lons)):
        lons_360[lon]=lons[lon]
        
    lons_360[len(lons)]=360
    
    #interpolate array to new grid point
    
    array_interpolated=np.zeros((len(lats),len(lons)+1))
    array_interpolated[:,0:len(lons)]=array
    array_interpolated[:,len(lons)]=array[:,0]
        

    return lats_90, lons_360, array_interpolated    
    

In [ ]:
def find_ozone_extremes(nc_fid, var, lev, years, extreme_years, model):
    
    # This function finds years with the highest and lowest ozone values in March and April 
    # based on daily mean ozone values.
    
    # The module requires the input of a daily averaged zm ozone file on pressure levels.
    
    # "nc_fid": input file including ozone variable
    # "years": number of years in input file 
    # "extreme_years": number of desired ozone extreme years.
    # "model": SOCOL, WACCM or MERRA
    # "var": name of ozone variable in input file
    # "lev": name of pressure level variable
    
    partial_column=True # if True: calculates ozone extremes based on 30-70 hPa partial ozone column; 
                        # if False: calculates ozone extremes based on 70 hPa O3 mixing ratio

    O3=nc_fid[var]
    plev=nc_fid[lev]
    time=nc_fid['time']
    
    O3=O3.interp(lat=np.linspace(-90,90,73))     # interpolate to 2.5°
    
    
    if partial_column ==True:
        
        #calculate partial ozone column from 30 to 70 hPa over the polar cap
    
        delta_p = np.zeros((len(O3.time), len(plev)))
    
        m_air = 28.964/(6.022E23)
        g = 980.6
            
        if plev[len(plev)-1] <= 1000 and model!='MERRA': # for pressure levels in hPa
            
            for level in range(1,len(plev)):
                delta_p[:,level].fill( plev[level] - plev[level-1])

            O3=O3.sel(lat=slice(60,90)) #average over polar cap
            
            weights = np.cos(np.deg2rad(O3.lat)) # latitudinal weights
            O3 = O3.weighted(weights)     
            O3=O3.mean(dim='lat')
            
            weights_p = xr.DataArray(delta_p*100, dims=['time','plev'], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = O3 * weights_p * 10/ (g * m_air)
            
            O3=O3.sel(plev=slice(30,70)) 
            O3 = O3.sum(dim='plev')
           
            O3 = O3/2.687E16  #calculate DU

            plev=plev.sel(plev=slice(30,70))
 
            
        if plev[len(plev)-1] > 1000: # for pressure levels in hPa
            
            for level in range(1,len(plev)):
                delta_p[:,level].fill( plev[level] - plev[level-1])  
                
            O3=O3.sel(lat=slice(60,90))  #average over polar cap  
            
            weights = np.cos(np.deg2rad(O3.lat)) # latitudinal weights
            O3 = O3.weighted(weights)     
   
            O3=O3.mean(dim='lat')
  
            weights_p = xr.DataArray(delta_p, dims=['time','plev'], coords=[time,plev])
            
            O3 = O3 * weights_p * 10/ (g * m_air)
            
            O3=O3.sel(plev=slice(3000,7000))
            O3 = O3.sum(dim='plev')
           
            O3 = O3/2.687E16 #calculate DU
            
            plev=plev.sel(plev=slice(3000,7000))
            plev=plev/100    
    
    
        if model=='MERRA':
            
            for level in range(0,len(plev)-1):
                delta_p[:,level].fill( plev[level] - plev[level+1])   
            
            O3=O3.sel(lat=slice(60,90))*28.970/47.9982 #average over polar cap and conversion to mol/mol
     
            weights = np.cos(np.deg2rad(O3.lat)) #latitudinal weights
            O3 = O3.weighted(weights)     
            O3=O3.mean(dim='lat')
            
            weights_p = xr.DataArray(delta_p*100, dims=['time','lev'], coords=[time,plev]) # difference between pressure levels in Pa
            
            O3 = O3 * weights_p * 10/ (g * m_air)
            
            O3=O3.sel(lev=slice(70,30)) 
            O3 = O3.sum(dim='lev')
           
            O3 = O3/2.687E16  #calculate DU

            plev=plev.sel(lev=slice(70,30))

        
    if partial_column==False:

        # average over polar cap. Choose values at 70hPa
        if plev[len(plev)-1] > 1000:
            O3=O3.sel(lat=slice(60,90),plev=7000)
            
        if plev[len(plev)-1] <= 1000 and model!='MERRA':
            O3=O3.sel(lat=slice(60,90),plev=70) 
            
        if model=='MERRA':
            O3=O3.sel(lat=slice(60,90),lev=70)
     
        weights = np.cos(np.deg2rad(O3.lat))
        weights.name = "weights"
        
        O3 = O3.weighted(weights)     
        
        O3=O3.mean(dim='lat')*1000000 # calculate ppmv
        
    
    # select March and April
    
   # O3=O3.groupby("time.dayofyear")-O3.groupby("time.dayofyear").mean() #detection based on anomalies
    var=O3
 
    
    O3_clim=O3.groupby("time.month").mean() #calculate monthly mean climatology
    
    #select values within March and April
    
    O3=O3.sel(time=nc_fid.time.dt.month.isin([3,4]))
    time=time.sel(time=nc_fid.time.dt.month.isin([3,4]))


    # select highest and lowest ozone values in each year

    O3_highest_values=O3.groupby("time.year").max() #select maximum ozone value in each spring
    O3_highest_indices=O3.groupby("time.year").argmax() #select index of maximum ozone value in each spring
    
    O3_lowest_values=O3.groupby("time.year").min() #select minimum ozone value in each spring
    O3_lowest_indices=O3.groupby("time.year").argmin() #select index of minimum ozone value in each spring
    
    time=time.groupby("time.year") 
    
    # find dates with the highest and lowest ozone values for each year
    
    O3_lowest_dates=[] # finds dates when ozone maximizes/minimizes each year
    O3_highest_dates=[]
    
    O3_years=np.zeros((years)) # list of all the years in the data

    for i,(year, group) in enumerate(time): #loop through every year and save date when lowest/highest ozone values occur

        O3_lowest_dates.append(np.array(group[O3_lowest_indices[i]]))
        O3_highest_dates.append(np.array(group[O3_highest_indices[i]]))
        O3_years[i]=year
    
    
    O3_highest_values=np.array(O3_highest_values)
    O3_lowest_values=np.array(O3_lowest_values)
    
    O3_highest_values=np.reshape(O3_highest_values, (years,))
    O3_lowest_values=np.reshape(O3_lowest_values, (years,))
    
    # find 50 highest and 50 lowest March and April ozone years based on daily values (this return the index of the respective year ozone extreme years)
    
    O3_highest=O3_highest_values.argsort()[-extreme_years:][::-1] #find 50 highest ozone values
    O3_lowest=O3_lowest_values.argsort()[0:extreme_years] #find 50 lowest ozone values

    # find ozone value and date of the 50 extreme ozone years

    O3_lowest_index=np.zeros((extreme_years)) # this saves the index when the ozone extreme maximizes in each respective year
    O3_highest_index=np.zeros((extreme_years))
    
    O3_lowest_date=[] # this saves the date vector when the ozone extreme maximizes in each respective year
    O3_highest_date=[]

    
    for i in range(extreme_years):
        O3_lowest_index[i]=int(O3_lowest_indices[O3_lowest[i]])
        O3_highest_index[i]=int(O3_highest_indices[O3_highest[i]])
        
        O3_lowest_date.append(O3_lowest_dates[O3_lowest[i]])
        O3_highest_date.append(O3_highest_dates[O3_highest[i]])
        
    
    O3_lowest_index=O3_lowest_index.astype(int)
    O3_highest_index=O3_highest_index.astype(int)
    
    
    print('Mean minimum ozone day: ' + str(np.mean(O3_lowest_index)) + ' ± ' + str(np.std(O3_lowest_index)))
    print('Mean maximum ozone day: ' + str(np.mean(O3_highest_index)) + ' ± ' + str(np.std(O3_highest_index)))
    

    O3_intersect=len(np.intersect1d(O3_highest, O3_lowest)) #this counts the number of years where there is both an ozone maximum and minimum

    #This function returns:
    
    #"O3_highest": indices of the maximum ozone years
    #"O3_lowest": indices of the minimum ozone years
    #np.reshape(O3_lowest_date, (extreme_years,)): dates of occurrence of the 50 strongest ozone minima
    #np.reshape(O3_highest_date, (extreme_years,)): dates of occurrence of the 50 strongest ozone maxima
    #np.reshape(O3_lowest_index, (extreme_years,)): index (days after March 1st) when strongest 50 ozone minima occur
    #np.reshape(O3_highest_index, (extreme_years,)): index (days after March 1st) when strongest 50 ozone maxima occur
    #"O3_intersect": number of years when both an ozone minimum and maximum occurs
    #"O3_years": year labels as in the netCDF input file

    return O3_highest, O3_lowest, np.reshape(O3_lowest_date, (extreme_years,)), np.reshape(O3_highest_date, (extreme_years,)), np.reshape(O3_lowest_index, (extreme_years,)), np.reshape(O3_highest_index, (extreme_years,)), O3_intersect, O3_years

In [ ]:
def analyse_ozone_extremes(nc_fid, var, extreme_years, surface_pressure,  O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years,model):

    # This function calculates and plots the surface impact (pressure, temperature or precipitation) for high and low 
    # ozone years.
    
    # It requires input in form of daily surface fiels (lat - lon) of the variable "var".
    # "extreme_years": number of desired ozone extreme years.
    # if "surface_pressure=True": the input variable is surface pressure in Pa. The variable will be divided by 100 to get result in hPa
    # "O3_highest_date": date of the ozone maxima
    # "O3_highest_date": date of the ozone minima
    # "O3_lowest_index": index (day after March 1st) when ozone minima occur
    # "O3_years": years in which ozone minima occur
    # "model": SOCOL, WACCM or MERRA
    
    var=nc_fid[var]
    
    lons=nc_fid['lon']
    lons=np.array(lons)
    lats=nc_fid['lat']
    lats=np.array(lats)
    time=nc_fid['time']
    time=np.array(time)
    
    # Calculate anomalies
 
    var_anomalies = var.groupby("time.dayofyear") - var.groupby("time.dayofyear").mean("time")
 
    # FIND INDICES OF OZONE EXTREMES   

    var_anomalies_xr=var_anomalies
    var_anomalies=np.array(var_anomalies)
    
    ozone_low_date=np.zeros((extreme_years))

    for year in range(extreme_years):
         ozone_low_date[year]=int(np.reshape(np.array(np.where(time==np.array(O3_lowest_date[year]))), (1,)))
        
    # get data following the 60 days after an ozone extreme event
    
    var_min=np.zeros((extreme_years,60,len(lats),len(lons)))

    for i in range(extreme_years):   
            var_min[i,:,:,:]=var_anomalies[int(ozone_low_date[i])-0:int(ozone_low_date[i]+60),:,:]
  
    # average over the 60-day period

    var_min = np.nanmean(var_min[:,:,:,:], axis=1)
 
    var_min_zm = np.nanmean(var_min, axis=0)

   # significance calculation switched off

    significance_low=[]
    
    # interpolate the values (to get values at the poles)
    
    if 90 and -90 not in lats:
        lats_90, lons_360, var_min_zm  = interpolate_pole(0, lons, lats, var_min_zm)
      
    else:
        lats_90, lons_360, var_min_zm  = interpolate_lons(0, lons, lats, var_min_zm)
 
    #new coordinates
    
    m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l')
    lons_m,lats_m=np.meshgrid(lons_360,lats_90)
    xpt,ypt = m(lons_m,lats_m)
    
    #function returns:
    
    #"var_min_zm": mean surface response in the 30 days following the ozone minima (lat-lon)
    # "xpt" & "ypt": coordinates used for plotting
    # "var_anomalies_xr": xarray containing the surface anomalies
    # "significance_low": significance of the surface response in low ozone years (lat-lon)
    
    #the output for high ozone years can be added here

    return var_min_zm, xpt,ypt, var_anomalies_xr, significance_low

In [ ]:
def calc_parc_o3(var):
    
        m_air = 28.964/(6.022E23)
        g = 980.6
    
        if 'plev' in var.dims: 
            plev=var.plev
            level='plev'
        if 'lev' in var.dims: 
            plev=var.lev
            level='lev'
        if 'level' in var.dims: 
            plev=var.level
            level='level'
        
        
        time=var.time
        delta_p = np.zeros((len(time), len(plev)))
        
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] <= 1000: 
            factor=100
            factor_2=1 # conversion Pa->hPa
        if plev[0]>plev[len(plev)-1] and plev[0] <=1000: 
            factor=100
            factor_2=1
        if plev[0]<plev[len(plev)-1] and plev[len(plev)-1] >1000: 
            factor=1
            factor_2=100
        if plev[0]>plev[len(plev)-1] and plev[0] >1000: 
            factor=1
            factor_2=100
        
        if plev[0]<plev[len(plev)-1]: # for pressure levels in hPa
            
            
            for levelx in range(1,len(plev)): delta_p[:,levelx].fill(plev[levelx] - plev[levelx-1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa
 
            O3 = var * weights_p * 10/ (g * m_air)
        
            if level=='plev': O3=O3.sel(plev=slice(30*factor_2,70*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(30*factor_2,70*factor_2))
            if level=='level': O3=O3.sel(level=slice(30*factor_2,70*factor_2))

            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
            
        if plev[0]>plev[len(plev)-1]: # for pressure levels in hPa
        
            
            for levelx in range(0,len(plev)-1): delta_p[:,levelx].fill(plev[levelx] - plev[levelx+1])

            weights_p = xr.DataArray(delta_p*factor, dims=['time',level], coords=[time,plev]) # difference between pressure levels in Pa

            O3 = var * weights_p * 10/ (g * m_air)
            
            if level =='plev': O3=O3.sel(plev=slice(70*factor_2,30*factor_2)) 
            if level=='lev': O3=O3.sel(lev=slice(70*factor_2,30*factor_2)) 
            if level=='level': O3=O3.sel(level=slice(70*factor_2,30*factor_2)) 
                
            O3 = O3.sum(dim=level)
            O3 = O3/2.687E16  #calculate DU
    
        return O3.where(O3 != 0)

Import timeslice 2000 dataset:

In [ ]:
#WACCM INT_O3

nc_fid=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc')

In [ ]:
O3=nc_fid['O3']

O3=calc_parc_o3(O3)

O3=O3.mean(dim='lon')

var=O3.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var.lat))
var = var.weighted(weights)     
var=var.mean(dim='lat')

In [ ]:
#WACCM INT_O3 PSL

nc_fid_SURF=xr.open_dataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.PSL.merged.nc')
PSL_clim=nc_fid_SURF['PSL']
PSL_clim=PSL_clim.groupby('time.dayofyear').mean()

In [ ]:
%reload_ext autoreload
%autoreload 2

fig, ax = plt.subplots(1,1,figsize=(13,10), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')

extreme_years=10
years = 39
model='WACCM'
lev='plev'

O3='O3'

O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes(nc_fid, O3,  lev, years,extreme_years,model)
var_min_slp_x,xpt,ypt, var_anomalies_slp_x, significance_slp_x = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)

O3_neutral_years=np.zeros((years-extreme_years))

var_clim=var.groupby("time.dayofyear").mean("time")

x=0

O3_lowest=O3_years[O3_lowest]

for y in range(years):

    if np.isin(O3_years[y],O3_lowest)==False:

        O3_neutral_years[x]=O3_years[y]
        x=x+1

var_lowest = var.sel(time=var.time.dt.year.isin([O3_lowest]))
var_lowest_mean = np.array(var_lowest.groupby("time.dayofyear").mean("time"))
var_lowest_std = np.array(var_lowest.groupby("time.dayofyear").std("time"))

var_lowest_before = var.sel(time=var.time.dt.year.isin([O3_lowest-1]))
var_lowest_mean_before = np.array(var_lowest_before.groupby("time.dayofyear").mean("time"))
var_lowest_std_before = np.array(var_lowest_before.groupby("time.dayofyear").std("time"))

var_highest = var.sel(time=var.time.dt.year.isin([O3_highest]))
var_highest_mean = np.array(var_highest.groupby("time.dayofyear").mean("time"))
var_highest_std = np.array(var_highest.groupby("time.dayofyear").std("time"))

var_highest_before = var.sel(time=var.time.dt.year.isin([O3_highest-1]))
var_highest_mean_before = np.array(var_highest_before.groupby("time.dayofyear").mean("time"))
var_highest_std_before = np.array(var_highest_before.groupby("time.dayofyear").std("time"))


var_neutral = var.sel(time=var.time.dt.year.isin([O3_neutral_years]))
var_neutral_mean = np.array(var_neutral.groupby("time.dayofyear").mean("time"))
var_neutral_std = np.array(var_neutral.groupby("time.dayofyear").std("time"))


var_neutral_before = var.sel(time=var.time.dt.year.isin([O3_neutral_years-1]))
var_neutral_mean_before = np.array(var_neutral_before.groupby("time.dayofyear").mean("time"))
var_neutral_std_before = np.array(var_neutral_before.groupby("time.dayofyear").std("time"))

var_lowest=np.reshape(np.array(var_lowest), (extreme_years, 365))
var_lowest_before=np.reshape(np.array(var_lowest_before), (extreme_years, 365))

color = cm.twilight(np.linspace(0, 1, 10))

for j in range(extreme_years):
    ax.plot(range(91,365), var_lowest[j,0:365-91],  color=color[j], alpha=0.8,linewidth=2, label='low O3 years')
    ax.plot(range(0,91), var_lowest_before[j,365-92:365-1] , color=color[j], alpha=0.8,linewidth=2)

#  plt.errorbar(range(92,len(var_highest_mean)), var_highest_mean[0:len(var_lowest_mean)-92] ,var_highest_std[0:len(var_lowest_mean)-92], color='red', alpha=0.5,elinewidth=1.5, linewidth=3, label='INT_O3, high O3')
plot=ax.errorbar(range(91,365), var_neutral_mean[0:365-91] ,var_neutral_std[0:365-91], color='grey', alpha=0.5,elinewidth=1.5, linewidth=3, label='all other years')


# plt.errorbar(range(0,92), var_highest_mean_before[len(var_lowest_mean)-92:len(var_lowest_mean)] ,var_highest_std_before[len(var_lowest_mean)-92:len(var_lowest_mean)], color='red', alpha=0.5,elinewidth=1.5, linewidth=3)
ax.errorbar(range(0,91), var_neutral_mean_before[365-92:365-1] ,var_neutral_std[365-92:365-1], color='grey', alpha=0.5,elinewidth=1.5, linewidth=3)

ax.set_xlim(0,366)

ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
ax.set_xticklabels( ['Oct', 'Nov', 'Dec', 'Jan','Feb','Mar','Apr','May','June','July','Aug','Sep'], fontsize=15)
#  ax[i].set_xticklabels( ['01.10.', '01.11.', '01.12.', '01.01.','01.02.','01.03.','01.04.','01.05.','01.06.','01.07.','01.08.','01.09.'], fontsize=15)

    
ax.set_ylabel('Partial ozone column, 30-70 hPa (DU)', fontsize=18)
ax.set_yticklabels(ax.get_yticks(), size = 18)

plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_10extr.pdf')
plt.savefig('/home/weiji/test code/plots/O3_evolution_min_restart_10extr.png')

Calculate ozone minimum years and plot ozone partial column in those years:

In [ ]:
m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l')
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-6,6, 16)

plot_slp=m.contourf(xpt,ypt,var_min_slp_x/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both')
#m.contourf(xpt, ypt,  significance_slp, colors='none', levels=[0,0.05, 100], hatches=['','....'])
plt.title('Mean surface response (n=10)')

In [ ]:
extreme_years=1

O3_highest, O3_lowest, O3_lowest_date, O3_highest_date, O3_lowest_index, O3_highest_index, O3_intersect, O3_years = find_ozone_extremes(nc_fid, O3,  lev, years,extreme_years,model)
var_min_slp_008,xpt,ypt, var_anomalies_slp_x, significance_slp_x = analyse_ozone_extremes(nc_fid_SURF, 'PSL', extreme_years, True, O3_highest_date, O3_lowest_date, O3_lowest_index, O3_years, model)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l')
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-4,4, 16)

plot_slp=m.contourf(xpt,ypt,var_min_slp_008/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both')
#m.contourf(xpt, ypt,  significance_slp, colors='none', levels=[0,0.05, 100], hatches=['','....'])
plt.title('surface response in year 008')

## SURFACE RESPONSE OF FEB INITIALISATION

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_F=nc_fid_F['O3']
o3_F=o3_F.mean(dim='lon')

o3_F=calc_parc_o3(o3_F)

var_F=o3_F.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_F.lat))
var_F = var_F.weighted(weights)     
var_F=var_F.mean(dim='lat')

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

o3_FNC=nc_fid_FNC['O3']
o3_FNC=o3_FNC.mean(dim='lon')

o3_FNC=calc_parc_o3(o3_FNC)

var_FNC=o3_FNC.sel(lat=slice(60,90))

weights = np.cos(np.deg2rad(var_FNC.lat))
var_FNC = var_FNC.weighted(weights)     
var_FNC=var_FNC.mean(dim='lat')

In [ ]:
nc_fid_F=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

p_F=nc_fid_F['PSL']

In [ ]:
var_F_runmean=var_F.rolling(time=5,center=True).mean() #calculate 5-day running mean
var_F_runmean=var_F_runmean.sel(time=var_F_runmean.time.dt.month.isin([3,4])) #select March/April
var_F_runmean=var_F_runmean.groupby('member').argmin(dim='time') #find date (day after March 1st) of minimum ozone value in each member
print(np.array(var_F_runmean))

In [ ]:
p_F=p_F.groupby("time.dayofyear")-PSL_clim #remove daily climatology of timeslice simulation to get anomalies

p_F_time=p_F.sel(time=p_F.time.dt.month.isin([3,4,5])) 

p_F_time=p_F_time.groupby('member')

PSL=[]

i=0
for member,group in p_F_time:
    PSL.append(np.array(np.mean(group[int(var_F_runmean[i]):int(var_F_runmean[i])+60,:,:],axis=0))) #for each member, average over 30 days after ozone minima
    i=i+1
    
PSL=np.nanmean(PSL, axis=0) #average over all members

In [ ]:
nc_fid_FNC=xr.open_mfdataset('/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_NOCOUPL_SURF/BWCN.e122.f19_g16.002.0008-02.*.nc' ,concat_dim='member', combine='nested')

p_FNC=nc_fid_FNC['PSL']

var_FNC_runmean=var_FNC.rolling(time=5,center=True).mean() #calculate 5-day running mean
var_FNC_runmean=var_FNC_runmean.sel(time=var_FNC_runmean.time.dt.month.isin([3,4])) #select March/April
var_FNC_runmean=var_FNC_runmean.groupby('member').argmin(dim='time') #find date (day after March 1st) of minimum ozone value in each member
print(np.array(var_FNC_runmean))
p_FNC=p_FNC.groupby("time.dayofyear")-PSL_clim #remove daily climatology of timeslice simulation to get anomalies

p_FNC_time=p_FNC.sel(time=p_FNC.time.dt.month.isin([3,4,5])) 

p_FNC_time=p_FNC_time.groupby('member')

PSL_NC=[]

i=0
for member,group in p_FNC_time:
    PSL_NC.append(np.array(np.mean(group[int(var_FNC_runmean[i]):int(var_FNC_runmean[i])+60,:,:],axis=0))) #for each member, average over 30 days after ozone minima
    i=i+1
    
PSL_NC=np.nanmean(PSL_NC, axis=0) #average over all members

In [ ]:
fig, ax = plt.subplots(1,4,figsize=(12,4), constrained_layout=True, edgecolor='k', sharex='col', sharey='row')

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[0])
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

levels = np.linspace(-6,6, 25)

plot_slp=m.contourf(xpt,ypt,var_min_slp_x/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[0])
ax[0].set_title('mean surface response \n after ozone minima \n (n=10)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[1])
m.drawcoastlines(linewidth=0.3)
m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)


plot_slp=m.contourf(xpt,ypt,var_min_slp_008/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[1])
ax[1].set_title('surface response \n in year 008 \n (n=1)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[2])
lons_m,lats_m=np.meshgrid(p_F.lon,p_F.lat)
xpt2,ypt2 = m(lons_m,lats_m)
m.drawcoastlines(linewidth=0.3)

m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)


plot_slp=m.contourf(xpt2,ypt2,PSL/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[2])
ax[2].set_title('surface response, Feb, \n small pertlim \n (n=30)', fontsize=18)

m = Basemap(projection='ortho', lat_0=90,lon_0=0,resolution='l', ax=ax[3])
lons_m,lats_m=np.meshgrid(p_F.lon,p_F.lat)
xpt2,ypt2 = m(lons_m,lats_m)
m.drawcoastlines(linewidth=0.3)

m.drawmeridians(np.arange(0,360,60), linewidth=0.3)
m.drawparallels(np.arange(-90,90,30), linewidth=0.3)

plot_slp=m.contourf(xpt2,ypt2,PSL_NC/100, cmap=plt.cm.get_cmap('RdBu_r'), interpolation='gaussian', levels=levels, extend='both', ax=ax[3])
ax[3].set_title('surface response, Feb, \n NOCOUPL \n (n=30)', fontsize=18)

cbar=fig.colorbar(plot_slp, ax=ax[0:4], location='bottom', shrink=0.5)
cbar.set_label(r"hPa",  fontsize=18)

plt.savefig('/home/weiji/test code/plots/slp_response_ozone_min_Feb.pdf', bbox_inches="tight")